# 01 — Load Tier 1 data

Read raw CSV/XLSX files. Print shapes and dtypes. Cache typed copies as parquet so downstream notebooks do not pay Excel-parse cost.

**Upstream:** raw files in `data/`

**Output:** parquets in `pipeline/artifacts/` (sales, cb, tpr, inv_snapshot, item_master, vendor, po, slprsn_key)

**Promotes to:** `src/load.py` once verified.

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'pipeline' else Path.cwd()
DATA = ROOT / 'data'
ART  = ROOT / 'pipeline' / 'artifacts'
ART.mkdir(parents=True, exist_ok=True)

SALES_CSV   = DATA / 'POP_SalesTransactionHistory.csv'
INV_XLSX    = DATA / 'POP_InventorySnapshot.xlsx'
ITEM_XLSX   = DATA / 'POP_ItemSpecMaster.xlsx'
VENDOR_XLSX = DATA / 'POP_VendorMaster.xlsx'
PO_XLSX     = DATA / 'POP_PurchaseOrderHistory.XLSX'
CB_XLSX     = DATA / 'POP_ChargeBack_Deductions_Penalties_Freight.xlsx'
SLPRSN_XLSX = DATA / 'SLPRSNID_SALESCHANNEL_KEY.xlsx'

for p in [SALES_CSV, INV_XLSX, ITEM_XLSX, VENDOR_XLSX, PO_XLSX, CB_XLSX, SLPRSN_XLSX]:
    assert p.exists(), f'missing: {p}'

## 2. Read raw files

One source-of-truth load per file. Downstream notebooks read the cached parquets in step 5.

In [2]:
sales = pd.read_csv(SALES_CSV, parse_dates=['DOCDATE'], low_memory=False)

inv = pd.concat([
    pd.read_excel(INV_XLSX, sheet_name='Site 1 - SF').assign(DC='SF'),
    pd.read_excel(INV_XLSX, sheet_name='Site 2 - NJ').assign(DC='NJ'),
    pd.read_excel(INV_XLSX, sheet_name='Site 3 - LA').assign(DC='LA'),
], ignore_index=True)

items   = pd.read_excel(ITEM_XLSX,   sheet_name='Item Spec Master')
vendors = pd.read_excel(VENDOR_XLSX, sheet_name='Supplier Master')
pos     = pd.read_excel(PO_XLSX,     sheet_name='PO Order History 2023-2025')
cb      = pd.read_excel(CB_XLSX,     sheet_name='Data - Deductions & Cause Code')
slprsn  = pd.read_excel(SLPRSN_XLSX)   # default first sheet; adjust if needed

for name, df in [('sales',sales),('inv',inv),('items',items),('vendors',vendors),('pos',pos),('cb',cb),('slprsn',slprsn)]:
    print(f'{name:10s} {df.shape}  cols[:6]={list(df.columns)[:6]}')

sales      (236818, 23)  cols[:6]=['LOCNCODE', 'SLPRSNID', 'CUSTNMBR', 'CITY', 'STATE', 'ZIPCODE']
inv        (219, 5)  cols[:6]=['Item Number', 'Description', 'Available', 'On Hand', 'DC']
items      (65, 15)  cols[:6]=['Item Number', 'Description', 'Case Pack', 'unit dimension (L*W*H) (in)', 'inner case dimension (L*W*H) (in)', 'master case dimension (L*W*H) (in)']
vendors    (73, 17)  cols[:6]=['Brand', 'Product Line', 'Category', 'Abbr. Legend (for WH)', 'Vendor ID', 'State']
pos        (5281, 16)  cols[:6]=['PO Number', 'PO Date', 'Required Date', 'Promised Ship Date', 'Receipt Date', 'POP Receipt Number']
cb         (18804, 13)  cols[:6]=['Location Code', 'Salesperson ID', 'Customer Number', 'City from Sales Transaction', 'State from Sales Transaction', 'SOP Type']
slprsn     (46, 3)  cols[:6]=['SLPRSNID', 'SALESCHANNEL', 'SALESCHANNEL_DESC']


## 3. Derive TPR subset of chargebacks

Chargebacks have many cause codes. The TPR / promo codes are what drive `is_promo`; split them out here so downstream notebooks can grab just `tpr.parquet`.

In [3]:
tpr_mask = cb['Cause Code Desc'].str.contains('TPR|promo|price reduction', case=False, na=False)
tpr = cb[tpr_mask].copy()

print(f'tpr rows: {len(tpr)} of {len(cb)} chargeback rows ({len(tpr)/len(cb):.1%})')
print()
print('Top cause codes in tpr subset:')
print(tpr['Cause Code'].value_counts().head(15))

tpr rows: 6868 of 18804 chargeback rows (36.5%)

Top cause codes in tpr subset:
Cause Code
CRED03      4113
CRED05      1645
CRED02       571
CRED04       259
CRED10-D     150
CRED-PRO     130
Name: count, dtype: int64


## 4. Validate

In [4]:
print('sales date range   :', sales['DOCDATE'].min(), '->', sales['DOCDATE'].max())
print('unique SKUs (sales):', sales['ITEMNMBR'].nunique())
print('unique SKUs (items):', items.iloc[:, 0].nunique())
print('inv rows per DC    :', inv['DC'].value_counts().to_dict())
print('tpr customers      :', tpr['Customer Number'].nunique() if 'Customer Number' in tpr.columns else 'col missing')
print()
print('sales columns with nulls (top 10):')
print(sales.isna().sum().sort_values(ascending=False).head(10))

sales date range   : 2023-01-03 00:00:00 -> 2026-04-13 00:00:00
unique SKUs (sales): 83
unique SKUs (items): 64
inv rows per DC    : {'SF': 73, 'NJ': 73, 'LA': 73}
tpr customers      : 88

sales columns with nulls (top 10):
ZIPCODE           4975
STATE             3876
CITY              3748
Product Type       490
Customer Type        4
SLPRSNID             4
LOCNCODE             0
XTNDPRCE_adj         0
UOM_Price            0
Margin_Pct_adj       0
dtype: int64


In [5]:
# --- Join coverage checks (prerequisite for downstream notebooks) ---
sales_skus = set(sales['ITEMNMBR'].dropna().unique())
item_skus  = set(items['Item Number'].dropna().unique())
inv_skus   = set(inv['Item Number'].dropna().unique())

# A. Sales SKUs not in item master → no lead time / supplier / cost available
orphans_sku = sales_skus - item_skus
orphan_rows = sales[sales['ITEMNMBR'].isin(orphans_sku)]
print(f'A. Sales SKUs with no item-master row : {len(orphans_sku)} / {len(sales_skus)}')
print(f'   → {len(orphan_rows):,} / {len(sales):,} sales rows ({len(orphan_rows)/len(sales):.1%})')
if orphans_sku:
    print(f'   sample: {sorted(orphans_sku)[:10]}')

# B. Salesperson IDs in sales not in the channel key → can't assign MM/AM/HF
sales_slp = set(sales['SLPRSNID'].dropna().astype(str).unique())
key_slp   = set(slprsn['SLPRSNID'].dropna().astype(str).unique())
unmapped_slp = sales_slp - key_slp
unmapped_rows = sales[sales['SLPRSNID'].astype(str).isin(unmapped_slp)]
print(f'\nB. Salesperson IDs in sales not in key: {len(unmapped_slp)} / {len(sales_slp)}')
print(f'   → {len(unmapped_rows):,} / {len(sales):,} sales rows ({len(unmapped_rows)/len(sales):.1%})')
if unmapped_slp:
    print(f'   unmapped: {sorted(unmapped_slp)}')

# C. Inventory coverage per DC
missing_inv_anywhere = sales_skus - inv_skus
print(f'\nC. Sales SKUs with no inventory row in ANY DC: {len(missing_inv_anywhere)} / {len(sales_skus)}')
if missing_inv_anywhere:
    print(f'   sample: {sorted(missing_inv_anywhere)[:10]}')
print('   Per-DC coverage:')
for dc in ['SF', 'NJ', 'LA']:
    dc_skus = set(inv.loc[inv['DC'] == dc, 'Item Number'].dropna().unique())
    covered = len(sales_skus & dc_skus)
    print(f'     {dc}: {len(dc_skus)} SKUs in inv | covers {covered}/{len(sales_skus)} sales SKUs ({covered/len(sales_skus):.1%})')

A. Sales SKUs with no item-master row : 37 / 83
   → 34,625 / 236,818 sales rows (14.6%)
   sample: ['A-61234', 'A-61250', 'A-61260', 'AC-B9SLE', 'D-13200N', 'D-13212', 'D-13220N', 'D-13300N', 'D-13306', 'D-13400']

B. Salesperson IDs in sales not in key: 0 / 46
   → 0 / 236,818 sales rows (0.0%)

C. Sales SKUs with no inventory row in ANY DC: 33 / 83
   sample: ['A-61234', 'A-61250', 'A-61260', 'AC-B16BK', 'AC-B6SLJ', 'D-13200N', 'D-13212', 'D-13220N', 'D-13300N', 'D-13306']
   Per-DC coverage:
     SF: 73 SKUs in inv | covers 50/83 sales SKUs (60.2%)
     NJ: 73 SKUs in inv | covers 50/83 sales SKUs (60.2%)
     LA: 73 SKUs in inv | covers 50/83 sales SKUs (60.2%)


## 5. Save downstream artifact

In [6]:
import pyarrow
print(f'pandas {pd.__version__}  pyarrow {pyarrow.__version__}')

# Smoke test: does parquet work at all on a trivial frame?
try:
    pd.DataFrame({'x': [1, 2, 3]}).to_parquet(ART / '_smoke.parquet')
    (ART / '_smoke.parquet').unlink()
    print('pyarrow smoke test: OK')
except Exception as e:
    print(f'pyarrow smoke test FAILED: {type(e).__name__}: {e}')
    print('  → try: pip install -U pyarrow "pandas>=2.0"')
print()

def _dedup(cols):
    seen, out, renamed = {}, [], []
    for c in cols:
        key = str(c)
        if key in seen:
            seen[key] += 1
            new = f'{key}__{seen[key]}'
            renamed.append((key, new))
            out.append(new)
        else:
            seen[key] = 0
            out.append(key)
    return out, renamed

def _coerce_mixed_object_cols(df):
    """Cast object-dtype columns with heterogeneous Python types to string.
    Pyarrow rejects mixed-type columns (e.g. Case Pack = int+str, Zip = int+str).
    NaN is preserved. `exclude='str'` skips pandas 3 native-string columns to silence
    the Pandas4Warning about select_dtypes('object') matching str columns."""
    renamed = []
    object_cols = df.select_dtypes(include='object', exclude='str').columns
    for col in object_cols:
        sample = df[col].dropna().head(200)
        types = {type(v).__name__ for v in sample}
        if len(types) > 1:
            df[col] = df[col].where(df[col].isna(), df[col].astype(str))
            renamed.append((col, sorted(types)))
    return renamed

to_cache = {
    'sales':         sales,
    'cb':            cb,
    'tpr':           tpr,
    'inv_snapshot':  inv,
    'item_master':   items,
    'vendor_master': vendors,
    'po':            pos,
    'slprsn_key':    slprsn,
}

for name, df in to_cache.items():
    df_out = df.copy()

    new_cols, dup_renamed = _dedup(df_out.columns)
    df_out.columns = new_cols
    if dup_renamed:
        print(f'{name:14s} deduped cols: {dup_renamed}')

    mixed = _coerce_mixed_object_cols(df_out)
    if mixed:
        print(f'{name:14s} cast mixed-type cols to str: {mixed}')

    path = ART / f'{name}.parquet'
    try:
        df_out.to_parquet(path)
        print(f'{name:14s} {df_out.shape} -> {path.name}')
    except Exception as e:
        alt = ART / f'{name}.pkl'
        df_out.to_pickle(alt)
        print(f'{name:14s} parquet FAILED ({type(e).__name__}): {e}')
        print(f'               saved pickle -> {alt.name}')

pandas 3.0.2  pyarrow 23.0.1
pyarrow smoke test: OK

sales          (236818, 23) -> sales.parquet
cb             (18804, 13) -> cb.parquet
tpr            (6868, 13) -> tpr.parquet
inv_snapshot   (219, 5) -> inv_snapshot.parquet
item_master    cast mixed-type cols to str: [('Case Pack', ['int', 'str']), ('unit dimension (L*W*H) (in)', ['int', 'str']), ('Case/ Pallet', ['int', 'str']), ('UPC#', ['int', 'str']), ('Shelf Life (Months)', ['int', 'str']), ('Lead Time', ['float', 'str'])]
item_master    (65, 15) -> item_master.parquet
vendor_master  cast mixed-type cols to str: [('Zip', ['int', 'str'])]
vendor_master  (73, 17) -> vendor_master.parquet
po             (5281, 16) -> po.parquet
slprsn_key     (46, 3) -> slprsn_key.parquet


## 6. Promote

Once loads and sanity checks look right, move the read logic into `src/load.py` as a single `load_all()` returning a dict of dataframes. Downstream dev notebooks can then:

```python
from src.load import load_all
data = load_all()
sales, cb, tpr = data['sales'], data['cb'], data['tpr']
```

Keep this notebook as the human-readable "how the data was parsed" reference.